# Forest Cover Type Classification

Multiclass prediction of seven forest cover types using cartographic attributes. The workflow includes domain-informed feature engineering, ablation analysis, baseline benchmarking, Optuna tuning, and a probability-averaged ensemble of LightGBM, XGBoost and ExtraTrees.

**Data:** [Kaggle Forest Cover Type Prediction](https://www.kaggle.com/competitions/forest-cover-type-prediction/data). Place `train.csv` and `test.csv` in the local `data/` directory before running.

## 1. Setup and data

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import optuna

SEED = 42
FAST_MODE = os.environ.get("FOREST_FAST_MODE", "0") == "1"
CV_FOLDS = 2 if FAST_MODE else 3
N_TRIALS = 1 if FAST_MODE else 20
np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR = Path("data")
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
if not test_path.exists() and (DATA_DIR / "test-full.csv").exists():
    test_path = DATA_DIR / "test-full.csv"
if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError("Download train.csv and test.csv from the Kaggle competition into data/.")
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
assert "Cover_Type" in train.columns and "Id" in train.columns and "Id" in test.columns
print("Training:", train.shape, "| Competition test:", test.shape)
print("Target distribution:")
print(train["Cover_Type"].value_counts().sort_index())

## 2. Feature engineering

In [ ]:
SOIL_ELU = np.array([
    2702, 2703, 2704, 2705, 2706, 2717, 3501, 3502, 4201, 4703,
    4704, 4744, 4758, 5101, 5151, 6101, 6102, 6731, 7101, 7102,
    7103, 7201, 7202, 7700, 7701, 7702, 7709, 7710, 7745, 7746,
    7755, 7756, 7757, 7790, 8703, 8707, 8708, 8771, 8772, 8776
], dtype=int)
GROUPS = ("aspect", "hydrology", "hillshade", "terrain", "ecology")

def engineer_features(frame, groups=GROUPS):
    frame = frame.drop(columns=["Id", "Cover_Type"], errors="ignore").copy()
    h = frame["Horizontal_Distance_To_Hydrology"].astype(float)
    v = frame["Vertical_Distance_To_Hydrology"].astype(float)
    elevation = frame["Elevation"].astype(float)
    slope = frame["Slope"].astype(float)
    road = frame["Horizontal_Distance_To_Roadways"].astype(float)
    fire = frame["Horizontal_Distance_To_Fire_Points"].astype(float)
    hydro_distance = np.hypot(h, v)

    if "aspect" in groups:
        radians = np.deg2rad(frame.pop("Aspect"))
        frame["Aspect_sin"] = np.sin(radians)
        frame["Aspect_cos"] = np.cos(radians)
    if "hydrology" in groups:
        frame["Hydrology_Euclidean"] = hydro_distance
        frame["Hydrology_Manhattan"] = h.abs() + v.abs()
        frame["Water_Elevation"] = elevation - v
    if "hillshade" in groups:
        shade = frame[["Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm"]]
        frame["Hillshade_mean"] = shade.mean(axis=1)
        frame["Hillshade_range"] = shade.max(axis=1) - shade.min(axis=1)
        frame["Hillshade_Asymmetry"] = frame["Hillshade_9am"] - frame["Hillshade_3pm"]
    if "terrain" in groups:
        frame["Elevation_sq"] = elevation**2
        frame["Slope_sq"] = slope**2
        frame["Elevation_x_Slope"] = elevation * slope
        frame["Elev_Hydro_ratio"] = elevation / (1.0 + hydro_distance)
        frame["Fire_Road_Sum"] = road + fire
        frame["Fire_Road_Diff"] = (road - fire).abs()
    if "ecology" in groups:
        elevation_band = pd.cut(elevation, [-np.inf, 2300, 2600, 2900, 3200, np.inf], labels=False).astype(int)
        wild_cols = [f"Wilderness_Area{i}" for i in range(1, 5)]
        soil_cols = [f"Soil_Type{i}" for i in range(1, 41)]
        assert (frame[wild_cols].sum(axis=1) == 1).all(), "Invalid one-hot wilderness indicators"
        assert (frame[soil_cols].sum(axis=1) == 1).all(), "Invalid one-hot soil indicators"
        wild_type = frame[wild_cols].to_numpy().argmax(axis=1) + 1
        soil_type = frame[soil_cols].to_numpy().argmax(axis=1)
        elu_code = SOIL_ELU[soil_type]
        frame["Elevation_band"] = elevation_band
        frame["Wilderness_Area_int"] = wild_type
        frame["Wild_x_ElevBand"] = 10 * wild_type + elevation_band
        frame["Soil_climatic_zone"] = elu_code // 1000
        frame["Soil_geologic_zone"] = (elu_code // 100) % 10
    return frame.drop(columns=["Soil_Type15"], errors="ignore")

FINAL_GROUPS = tuple(g for g in GROUPS if g != "aspect")
y = train["Cover_Type"].astype(int) - 1
X = engineer_features(train, FINAL_GROUPS)
X_test = engineer_features(test, FINAL_GROUPS).reindex(columns=X.columns)
assert not X.isna().any().any() and not X_test.isna().any().any()
assert list(X.columns) == list(X_test.columns)
X_dev, X_hold, y_dev, y_hold = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
print("Final features:", X.shape[1], "| Development:", X_dev.shape, "| Holdout:", X_hold.shape)

Feature groups include hydrology distances, hillshade summaries, nonlinear terrain terms, wilderness/elevation interactions and ELU-based soil zone codes. The final model retains the raw `Aspect` variable, consistent with the ablation-based decision to exclude its circular encoding.

The soil climatic and geologic zones use the first and second digits of the published USFS ELU codes.

## 3. Feature ablation

In [ ]:
ablation = [("Cleaned baseline", ())]
for label, groups in [
    ("+ Aspect encoding", ("aspect",)),
    ("+ Hydrology", ("aspect", "hydrology")),
    ("+ Hillshade", ("aspect", "hydrology", "hillshade")),
    ("+ Terrain", ("aspect", "hydrology", "hillshade", "terrain")),
    ("+ Ecology", GROUPS),
    ("Final (without Aspect encoding)", FINAL_GROUPS),
]:
    ablation.append((label, groups))

ablation_rows = []
for label, groups in ablation:
    Xa = engineer_features(train, groups).iloc[X_dev.index]
    estimator = LGBMClassifier(
        n_estimators=40 if FAST_MODE else 250, learning_rate=0.07,
        num_leaves=31, random_state=SEED, verbose=-1, n_jobs=1
    )
    scores = cross_val_score(estimator, Xa, y_dev, cv=cv, scoring="accuracy", n_jobs=1)
    ablation_rows.append({"Configuration": label, "Features": Xa.shape[1],
                         "CV accuracy": float(scores.mean()), "CV std": float(scores.std())})
ablation_results = pd.DataFrame(ablation_rows)
display(ablation_results.round(4))

## 4. Model-family comparison

In [ ]:
tree_size = 40 if FAST_MODE else 250
baseline_models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic (L2)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
    "Logistic (Elastic Net)": make_pipeline(StandardScaler(), LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=2000, random_state=SEED)),
    "Random Forest": RandomForestClassifier(n_estimators=tree_size, class_weight="balanced_subsample", n_jobs=1, random_state=SEED),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=tree_size, class_weight="balanced", n_jobs=1, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=tree_size, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=tree_size, max_depth=7, learning_rate=0.07,
                             objective="multi:softprob", eval_metric="mlogloss", n_jobs=1, random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=tree_size, learning_rate=0.07, num_leaves=63,
                               verbose=-1, n_jobs=1, random_state=SEED),
}
rows = []
for name, model in baseline_models.items():
    scores = cross_val_score(model, X_dev, y_dev, cv=cv, scoring="accuracy", n_jobs=1)
    rows.append({"Model": name, "CV accuracy": scores.mean(), "CV std": scores.std()})
baseline_results = pd.DataFrame(rows).sort_values("CV accuracy", ascending=False)
display(baseline_results.round(4))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(baseline_results["Model"][::-1], baseline_results["CV accuracy"][::-1])
ax.set(xlabel="Cross-validated accuracy", title="Baseline model comparison")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "baseline_comparison.png", dpi=150)
plt.show()

## 5. Optuna hyperparameter tuning

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
inner_tr, inner_val = next(splitter.split(X_dev, y_dev))
X_inner_tr, y_inner_tr = X_dev.iloc[inner_tr], y_dev.iloc[inner_tr]
X_inner_val, y_inner_val = X_dev.iloc[inner_val], y_dev.iloc[inner_val]

def instantiate_model(name, params):
    if name == "LightGBM":
        return LGBMClassifier(**params, objective="multiclass", random_state=SEED, n_jobs=1, verbose=-1)
    if name == "XGBoost":
        return XGBClassifier(**params, objective="multi:softprob", eval_metric="mlogloss", random_state=SEED, n_jobs=1)
    return ExtraTreesClassifier(**params, random_state=SEED, n_jobs=1)

def objective(name):
    def score(trial):
        if name == "LightGBM":
            params = dict(n_estimators=trial.suggest_int("n_estimators", 30 if FAST_MODE else 150, 60 if FAST_MODE else 450),
                          learning_rate=trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
                          num_leaves=trial.suggest_int("num_leaves", 15, 100),
                          max_depth=trial.suggest_int("max_depth", 5, 14),
                          colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
                          subsample=trial.suggest_float("subsample", 0.6, 1.0),
                          min_child_samples=trial.suggest_int("min_child_samples", 5, 40))
        elif name == "XGBoost":
            params = dict(n_estimators=trial.suggest_int("n_estimators", 30 if FAST_MODE else 150, 60 if FAST_MODE else 450),
                          learning_rate=trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
                          max_depth=trial.suggest_int("max_depth", 4, 10),
                          subsample=trial.suggest_float("subsample", 0.6, 1.0),
                          colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
                          reg_lambda=trial.suggest_float("reg_lambda", 0.1, 10, log=True))
        else:
            params = dict(n_estimators=trial.suggest_int("n_estimators", 30 if FAST_MODE else 150, 60 if FAST_MODE else 400),
                          max_depth=trial.suggest_categorical("max_depth", [None, 20, 35, 50]),
                          min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
                          max_features=trial.suggest_categorical("max_features", ["sqrt", 0.5, 0.8]))
        model = instantiate_model(name, params)
        model.fit(X_inner_tr, y_inner_tr)
        return accuracy_score(y_inner_val, model.predict(X_inner_val))
    return score

studies = {}
for name in ["LightGBM", "XGBoost", "ExtraTrees"]:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective(name), n_trials=N_TRIALS)
    studies[name] = study
    print(f"{name}: inner validation accuracy {study.best_value:.4f}")
fig, axs = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, study) in zip(axs, studies.items()):
    values = [t.value for t in study.trials if t.value is not None]
    ax.plot(np.arange(1, len(values)+1), np.maximum.accumulate(values))
    ax.set(xlabel="Trial", ylabel="Best validation accuracy", title=name)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "optuna_convergence.png", dpi=150)
plt.show()

## 6. Holdout evaluation and soft voting

In [ ]:
models = {}
for name, study in studies.items():
    model = instantiate_model(name, study.best_params)
    model.fit(X_dev, y_dev)
    models[name] = model

def ensemble_probabilities(fitted_models, features):
    classes = np.arange(7)
    arrays = []
    for model in fitted_models.values():
        proba = model.predict_proba(features)
        aligned = np.zeros((len(features), len(classes)), dtype=float)
        aligned[:, np.asarray(model.classes_, dtype=int)] = proba
        arrays.append(aligned)
    return np.mean(arrays, axis=0)

proba = ensemble_probabilities(models, X_hold)
pred = proba.argmax(axis=1)
result_rows = [{"Model": name, "Holdout accuracy": accuracy_score(y_hold, model.predict(X_hold))}
               for name, model in models.items()]
result_rows.append({"Model": "Soft-voting ensemble", "Holdout accuracy": accuracy_score(y_hold, pred)})
display(pd.DataFrame(result_rows).round(4))
class_names = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow", "Aspen", "Douglas-fir", "Krummholz"]
print(classification_report(y_hold, pred, target_names=class_names, digits=3))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(confusion_matrix(y_hold, pred, labels=np.arange(7)), annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set(xlabel="Predicted class", ylabel="True class", title="Soft-voting ensemble confusion matrix")
ax.set_xticklabels(np.arange(1,8))
ax.set_yticklabels(np.arange(1,8), rotation=0)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

## 7. Competition predictions

In [ ]:
final_models = {}
for name, study in studies.items():
    model = instantiate_model(name, study.best_params)
    model.fit(X, y)
    final_models[name] = model
pred_test = ensemble_probabilities(final_models, X_test).argmax(axis=1) + 1
submission = pd.DataFrame({"Id": test["Id"].to_numpy(), "Cover_Type": pred_test})
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
print(f"Saved {len(submission):,} predictions to {OUTPUT_DIR / 'submission.csv'}")
display(submission.head())